## End-to-End ML Workflow

End-to-End ML Workflow focuses on designing a complete machine-learning solution from raw data to model evaluation and responsible deployment. It requires a Data Scientist to consider not only the model, but also data preparation, feature engineering, train/test methodology, evaluation, limitations, monitoring, and ethical considerations.
Building on Project 08: Customer Segmentation and Project 09: AI Use-Case Design, we will implement a content-based movie recommendation system. The model will learn movie similarity from genres and recommend movies similar to those a user has rated highly.

>`Goal`: Build, evaluate, and document a complete recommendation ML workflow that can later be extended into a production-grade hybrid recommendation system.

In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

import warnings
warnings.filterwarnings("ignore")

In [2]:
DATA_DIR = Path("../Datasets_MLmodels/ml-latest-small")
P08_DIR = Path("../outputs/recommendation/project08_customer_segmentation")

P09_DIR = Path("../outputs/recommendation/project09_ai_use_case")
OUTPUT_DIR = Path("../outputs/recommendation/project10_end_to_end_ml")

FIGURE_DIR = OUTPUT_DIR / "figures"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
movies = pd.read_csv(DATA_DIR / "movies.csv")
ratings = pd.read_csv(DATA_DIR / "ratings.csv")
tags = pd.read_csv(DATA_DIR / "tags.csv")
links = pd.read_csv(DATA_DIR / "links.csv")

user_segments = pd.read_csv(P08_DIR / "user_segments.csv")

In [4]:
print("Movies :", movies.shape)
print("Ratings:", ratings.shape)
print("Tags   :", tags.shape)
print("Links  :", links.shape)
print("Users  :", user_segments.shape)

Movies : (9742, 3)
Ratings: (100836, 4)
Tags   : (3683, 4)
Links  : (9742, 3)
Users  : (610, 27)


In [5]:
print("Movies missing:")
print(movies.isnull().sum())

print("\nRatings missing:")
print(ratings.isnull().sum())

print("\nTags missing:")
print(tags.isnull().sum())

print("\nLinks missing:")
print(links.isnull().sum())

Movies missing:
movieId    0
title      0
genres     0
dtype: int64

Ratings missing:
userId       0
movieId      0
rating       0
timestamp    0
dtype: int64

Tags missing:
userId       0
movieId      0
tag          0
timestamp    0
dtype: int64

Links missing:
movieId    0
imdbId     0
tmdbId     8
dtype: int64


In [6]:
print("Movies duplicates :", movies.duplicated().sum())
print("Ratings duplicates:", ratings.duplicated().sum())
print("Tags duplicates   :", tags.duplicated().sum())
print("Links duplicates  :", links.duplicated().sum())

Movies duplicates : 0
Ratings duplicates: 0
Tags duplicates   : 0
Links duplicates  : 0


In [7]:
movies["genre_text"] = (movies["genres"].str.replace("|", " ", regex=False))
movies[["title", "genres", "genre_text"]].head()

,title,genres,genre_text
0,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,Adventure Animation Children Comedy Fantasy
1,Jumanji (1995),Adventure|Children|Fantasy,Adventure Children Fantasy
2,Grumpier Old Men (1995),Comedy|Romance,Comedy Romance
3,Waiting to Exhale (1995),Comedy|Drama|Romance,Comedy Drama Romance
4,Father of the Bride Part II (1995),Comedy,Comedy


In [8]:
tfidf = TfidfVectorizer()

movie_matrix = tfidf.fit_transform(movies["genre_text"])
movie_matrix.shape

(9742, 24)

In [9]:
similarity_matrix = cosine_similarity(movie_matrix)
similarity_matrix.shape

(9742, 9742)

In [10]:
movie_indices = pd.Series(movies.index, index=movies["movieId"]).drop_duplicates()

In [11]:
def recommend_movies(movie_id, n=10):
    if movie_id not in movie_indices:
        return pd.DataFrame()
    idx = movie_indices[movie_id]

    similarity_scores = similarity_matrix[idx]
    similar_indices = (similarity_scores.argsort()[::-1])
    similar_indices = [i for i in similar_indices if i != idx][:n]

    recommendations = movies.iloc[similar_indices][["movieId", "title", "genres"]].copy()
    recommendations["similarity"] = (similarity_scores[similar_indices])
    return recommendations

In [12]:
movies[movies["title"].str.contains("Toy Story", case=False, na=False)][["movieId", "title", "genres"]]

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
2355,3114,Toy Story 2 (1999),Adventure|Animation|Children|Comedy|Fantasy
7355,78499,Toy Story 3 (2010),Adventure|Animation|Children|Comedy|Fantasy|IMAX


In [13]:
recommend_movies(movie_id=1, n=10)

,movieId,title,genres,similarity
3000,4016,"Emperor's New Groove, The (2000)",Adventure|Animation|Children|Comedy|Fantasy,1.0
1706,2294,Antz (1998),Adventure|Animation|Children|Comedy|Fantasy,1.0
2809,3754,"Adventures of Rocky and Bullwinkle, The (2000)",Adventure|Animation|Children|Comedy|Fantasy,1.0
6194,45074,"Wild, The (2006)",Adventure|Animation|Children|Comedy|Fantasy,1.0
7760,91355,Asterix and the Vikings (Astérix et les Viking...,Adventure|Animation|Children|Comedy|Fantasy,1.0
8219,103755,Turbo (2013),Adventure|Animation|Children|Comedy|Fantasy,1.0
2355,3114,Toy Story 2 (1999),Adventure|Animation|Children|Comedy|Fantasy,1.0
6486,53121,Shrek the Third (2007),Adventure|Animation|Children|Comedy|Fantasy,1.0
9430,166461,Moana (2016),Adventure|Animation|Children|Comedy|Fantasy,1.0
8927,136016,The Good Dinosaur (2015),Adventure|Animation|Children|Comedy|Fantasy,1.0


In [14]:
def recommend_for_user(user_id, n=10, min_rating=4):
    user_ratings = ratings[(ratings["userId"] == user_id) &(ratings["rating"] >= min_rating)]
    if user_ratings.empty:
        return pd.DataFrame()
    liked_movies = user_ratings["movieId"].tolist()
    scores = np.zeros(len(movies))

    for movie_id in liked_movies:
        if movie_id not in movie_indices:
            continue
        idx = movie_indices[movie_id]
        scores += similarity_matrix[idx]
    rated_movies = set(ratings.loc[ratings["userId"] == user_id, "movieId"])
    scores[movies["movieId"].isin(rated_movies).values] = -1
    
    top_indices = (scores.argsort()[::-1])[:n]

    recommendations = movies.iloc[top_indices][["movieId", "title", "genres"]].copy()
    recommendations["score"] = (scores[top_indices])
    return recommendations

In [15]:
recommend_for_user(user_id=1, n=10)

,movieId,title,genres,score
8597,117646,Dragonheart 2: A New Beginning (2000),Action|Adventure|Comedy|Drama|Fantasy|Thriller,73.292981
4681,6990,The Great Train Robbery (1978),Action|Adventure|Comedy|Crime|Drama,71.571045
4005,5657,Flashback (1990),Action|Adventure|Comedy|Crime|Drama,71.571045
6570,55116,"Hunting Party, The (2007)",Action|Adventure|Comedy|Drama|Thriller,71.227169
3608,4956,"Stunt Man, The (1980)",Action|Adventure|Comedy|Drama|Romance|Thriller,70.039974
5471,26184,"Diamond Arm, The (Brilliantovaya ruka) (1968)",Action|Adventure|Comedy|Crime|Thriller,69.652946
4409,6503,Charlie's Angels: Full Throttle (2003),Action|Adventure|Comedy|Crime|Thriller,69.652946
5379,8968,After the Sunset (2004),Action|Adventure|Comedy|Crime|Thriller,69.652946
7409,80219,Machete (2010),Action|Adventure|Comedy|Crime|Thriller,69.652946
6774,60074,Hancock (2008),Action|Adventure|Comedy|Crime|Fantasy,69.509599


In [16]:
user_segments[["userId", "cluster"]].head()

,userId,cluster
0,1,3
1,2,1
2,3,3
3,4,1
4,5,0


In [17]:
user_segments[user_segments["userId"] == 1][["userId", "cluster"]]

,userId,cluster
0,1,3


In [18]:
example_user = 1
example_recommendations = recommend_for_user(user_id=example_user, n=10)

example_recommendations

,movieId,title,genres,score
8597,117646,Dragonheart 2: A New Beginning (2000),Action|Adventure|Comedy|Drama|Fantasy|Thriller,73.292981
4681,6990,The Great Train Robbery (1978),Action|Adventure|Comedy|Crime|Drama,71.571045
4005,5657,Flashback (1990),Action|Adventure|Comedy|Crime|Drama,71.571045
6570,55116,"Hunting Party, The (2007)",Action|Adventure|Comedy|Drama|Thriller,71.227169
3608,4956,"Stunt Man, The (1980)",Action|Adventure|Comedy|Drama|Romance|Thriller,70.039974
5471,26184,"Diamond Arm, The (Brilliantovaya ruka) (1968)",Action|Adventure|Comedy|Crime|Thriller,69.652946
4409,6503,Charlie's Angels: Full Throttle (2003),Action|Adventure|Comedy|Crime|Thriller,69.652946
5379,8968,After the Sunset (2004),Action|Adventure|Comedy|Crime|Thriller,69.652946
7409,80219,Machete (2010),Action|Adventure|Comedy|Crime|Thriller,69.652946
6774,60074,Hancock (2008),Action|Adventure|Comedy|Crime|Fantasy,69.509599


In [19]:
example_recommendations.to_csv(OUTPUT_DIR / "example_recommendations.csv", index=False)

#### Evaluation of the Recommendation Model: 
A recommendation system cannot be judged simply by saying:

>"The recommendations look good."

We need to test whether movies that a user actually liked can be recovered from the user's historical behavior.

In [20]:
user_rating_counts = (ratings.groupby("userId").size())

eligible_users = user_rating_counts[user_rating_counts >= 10].index

evaluation_ratings = ratings[ratings["userId"].isin(eligible_users)].copy()

In [21]:
evaluation_ratings = (
    evaluation_ratings
    .sort_values(["userId", "timestamp"])
)

test_ratings = (evaluation_ratings.groupby("userId").tail(1))

train_ratings = (evaluation_ratings.drop(test_ratings.index))

print("Training ratings:", train_ratings.shape)
print("Test ratings    :", test_ratings.shape)

Training ratings: (100226, 4)
Test ratings    : (610, 4)


In [22]:
test_positive = test_ratings[test_ratings["rating"] >= 4]

In [23]:
def recommend_from_training(user_id, train_data, n=10, min_rating=4):
    user_ratings = train_data[
        (train_data["userId"] == user_id) &
        (train_data["rating"] >= min_rating)
    ]
    if user_ratings.empty:
        return []
    liked_movies = user_ratings["movieId"].tolist()
    scores = np.zeros(len(movies))

    for movie_id in liked_movies:
        if movie_id not in movie_indices:
            continue
        idx = movie_indices[movie_id]
        scores += similarity_matrix[idx]
    rated_movies = set(train_data.loc[train_data["userId"] == user_id, "movieId"])
    scores[movies["movieId"].isin(rated_movies).values] = -1
    top_indices = (scores.argsort()[::-1])[:n]

    return movies.iloc[top_indices]["movieId"].tolist()

In [24]:
precision_scores = []
for _, row in test_positive.iterrows():
    user_id = row["userId"]
    actual_movie = row["movieId"]
    recommendations = recommend_from_training(user_id, train_ratings, n=10)

    if actual_movie in recommendations:
        precision_scores.append(1 / 10)
    else:
        precision_scores.append(0)

In [25]:
precision_at_10 = np.mean(precision_scores)

print(f"Precision@10: {precision_at_10:.4f}")

Precision@10: 0.0006


In [26]:
hits = []
for _, row in test_positive.iterrows():
    user_id = row["userId"]
    actual_movie = row["movieId"]
    recommendations = recommend_from_training(user_id, train_ratings, n=10)
    hits.append(int(actual_movie in recommendations))
hit_rate_at_10 = np.mean(hits)

print(f"Hit Rate@10: {hit_rate_at_10:.4f}")    

Hit Rate@10: 0.0055


In [27]:
evaluation_results = pd.DataFrame({
    "Metric": ["Precision@10", "Hit Rate@10"],
    "Value": [precision_at_10, hit_rate_at_10]
})

evaluation_results

,Metric,Value
0,Precision@10,0.000551
1,Hit Rate@10,0.005510


In [28]:
evaluation_results.to_csv(OUTPUT_DIR / "model_evaluation.csv", index=False)

Precision@10 measures how effectively the recommendation
list contains relevant items.

Hit Rate@10 measures how often the system successfully
places a hidden relevant movie within the top ten results.

Higher values indicate better recommendation performance,
but the metrics should be interpreted alongside diversity,
coverage and business objectives.

In [29]:
recommended_items = set()
sample_users = (eligible_users[:100])

for user_id in sample_users:
    recommendations = recommend_from_training(user_id, train_ratings, n=10)
    recommended_items.update(recommendations)
coverage = (
    len(recommended_items) / len(movies)
)
print(f"Catalog Coverage: {coverage:.4f}")    

Catalog Coverage: 0.0493


In [30]:
final_evaluation = pd.DataFrame({
    "Metric": ["Precision@10", "Hit Rate@10", "Catalog Coverage"],
    "Value": [precision_at_10, hit_rate_at_10, coverage]
})

final_evaluation

,Metric,Value
0,Precision@10,0.000551
1,Hit Rate@10,0.005510
2,Catalog Coverage,0.049271


In [31]:
final_evaluation.to_csv(OUTPUT_DIR / "final_evaluation.csv", index=False)

In [32]:
limitations = pd.DataFrame({
    "Limitation": [
        "Cold Start",
        "Genre-only Content Features",
        "Sparse User Behavior",
        "No Real-Time Feedback",
        "Limited Diversity Optimization",
        "Offline Evaluation"
    ],
    "Impact": [
        "New users have little history",
        "Movies with similar genres may still differ substantially",
        "Many users interact with relatively few movies",
        "Model does not immediately learn from new interactions",
        "Similar recommendations may dominate",
        "Offline results may differ from real user behavior"
    ]
})

limitations

,Limitation,Impact
0,Cold Start,New users have little history
1,Genre-only Content Features,Movies with similar genres may still differ su...
2,Sparse User Behavior,Many users interact with relatively few movies
3,No Real-Time Feedback,Model does not immediately learn from new inte...
4,Limited Diversity Optimization,Similar recommendations may dominate
5,Offline Evaluation,Offline results may differ from real user beha...


In [33]:
improvements = pd.DataFrame({
    "Improvement": [
        "Hybrid Recommendation",
        "Collaborative Filtering",
        "Better Movie Features",
        "Diversity Re-ranking",
        "Real-Time Feedback",
        "A/B Testing"
    ],
    "Purpose": [
        "Combine content and behavioral signals",
        "Learn from similar users",
        "Use tags and richer metadata",
        "Increase recommendation variety",
        "Adapt to changing preferences",
        "Measure real-world business impact"
    ]
})

improvements

,Improvement,Purpose
0,Hybrid Recommendation,Combine content and behavioral signals
1,Collaborative Filtering,Learn from similar users
2,Better Movie Features,Use tags and richer metadata
3,Diversity Re-ranking,Increase recommendation variety
4,Real-Time Feedback,Adapt to changing preferences
5,A/B Testing,Measure real-world business impact


In [34]:
production_layers = pd.DataFrame({
    "Layer": [
        "Data Collection",
        "Data Processing",
        "Feature Engineering",
        "Model Training",
        "Model Evaluation",
        "Deployment",
        "Monitoring",
        "Retraining"
    ],
    "Responsibility": [
        "Collect ratings and interactions",
        "Validate and clean data",
        "Create user/movie features",
        "Train recommendation model",
        "Measure recommendation quality",
        "Serve recommendations",
        "Track performance and drift",
        "Update model with new data"
    ]
})

production_layers

,Layer,Responsibility
0,Data Collection,Collect ratings and interactions
1,Data Processing,Validate and clean data
2,Feature Engineering,Create user/movie features
3,Model Training,Train recommendation model
4,Model Evaluation,Measure recommendation quality
5,Deployment,Serve recommendations
6,Monitoring,Track performance and drift
7,Retraining,Update model with new data


In [35]:
monitoring_metrics = pd.DataFrame({
    "Monitoring Area": [
        "Recommendation Quality",
        "Data Drift",
        "User Engagement",
        "Coverage",
        "Diversity",
        "System Performance",
        "Bias"
    ],
    "Example Metric": [
        "Precision@10",
        "Feature distribution changes",
        "Click/interaction rate",
        "Catalog coverage",
        "Average recommendation diversity",
        "Response latency",
        "Exposure across content groups"
    ]
})

monitoring_metrics

,Monitoring Area,Example Metric
0,Recommendation Quality,Precision@10
1,Data Drift,Feature distribution changes
2,User Engagement,Click/interaction rate
3,Coverage,Catalog coverage
4,Diversity,Average recommendation diversity
5,System Performance,Response latency
6,Bias,Exposure across content groups


In [36]:
ethical_controls = pd.DataFrame({
    "Risk": [
        "Privacy",
        "Popularity Bias",
        "Filter Bubble",
        "Lack of Transparency",
        "Feedback Loops",
        "Security"
    ],
    "Control": [
        "Minimize and protect user data",
        "Monitor exposure across catalog",
        "Introduce recommendation diversity",
        "Provide understandable explanations",
        "Continuously evaluate recommendation patterns",
        "Secure stored and transmitted data"
    ]
})

ethical_controls

,Risk,Control
0,Privacy,Minimize and protect user data
1,Popularity Bias,Monitor exposure across catalog
2,Filter Bubble,Introduce recommendation diversity
3,Lack of Transparency,Provide understandable explanations
4,Feedback Loops,Continuously evaluate recommendation patterns
5,Security,Secure stored and transmitted data


In [37]:
model_card = {
    "Model": "Content-Based Movie Recommendation System",
    "Dataset": "MovieLens ml-latest-small",
    "Primary_Features": "Movie genres",
    "Algorithm": "TF-IDF + Cosine Similarity",
    "Recommendation_Type": "Content-Based",
    "Output": "Top-N movie recommendations",
    "Evaluation": "Precision@10, Hit Rate@10, Catalog Coverage",
    "Primary_Limitation": "Does not fully capture collaborative user behavior",
    "Future_Improvement": "Hybrid recommendation system",
    "Ethical_Considerations": "Privacy, bias, diversity, transparency"
}

model_card_df = pd.DataFrame(model_card.items(), columns=["Component", "Description"])

model_card_df

,Component,Description
0,Model,Content-Based Movie Recommendation System
1,Dataset,MovieLens ml-latest-small
2,Primary_Features,Movie genres
3,Algorithm,TF-IDF + Cosine Similarity
4,Recommendation_Type,Content-Based
5,Output,Top-N movie recommendations
6,Evaluation,"Precision@10, Hit Rate@10, Catalog Coverage"
7,Primary_Limitation,Does not fully capture collaborative user beha...
8,Future_Improvement,Hybrid recommendation system
9,Ethical_Considerations,"Privacy, bias, diversity, transparency"


In [38]:
model_card_df.to_csv(OUTPUT_DIR / "model_card.csv", index=False)

production_layers.to_csv(OUTPUT_DIR / "production_workflow.csv", index=False)

monitoring_metrics.to_csv(OUTPUT_DIR / "monitoring_metrics.csv", index=False)

ethical_controls.to_csv(OUTPUT_DIR / "ethical_controls.csv", index=False)

limitations.to_csv(OUTPUT_DIR / "model_limitations.csv", index=False)

improvements.to_csv(OUTPUT_DIR / "model_improvements.csv",
 index=False
)

In [39]:
print("Project 10 outputs:\n")

for file in OUTPUT_DIR.iterdir():

    if file.is_file():
        print(file.name)

Project 10 outputs:

ethical_controls.csv
example_recommendations.csv
final_evaluation.csv
model_card.csv
model_evaluation.csv
model_improvements.csv
model_limitations.csv
monitoring_metrics.csv
production_workflow.csv
